# NFL Draft Pillar Scoring v2 — Section-Aware with Term Attribution

**Improvements over v1:**

| Problem in v1 | Fix in v2 |
|---|---|
| Only `strengths` text was scored | All three sections processed separately |
| Weakness terms inflated scores | `weaknesses` score subtracted from final |
| No explainability | Top driver/detractor terms stored per player per pillar |

**Final score formula:**
```
raw_pillar = sim(strengths) - sim(weaknesses) + 0.5 * sim(overview)
```
Then min-max scaled 0–100 per pillar.

## 0. Setup & Controls

In [98]:
import re
import warnings
import numpy as np
import pandas as pd
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

import scipy.sparse as sp

for resource in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(resource, quiet=True)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)

# ── Controls ───────────────────────────────────────────────────────────────────
PMI_TOP_N    = 30    # auto-discovered bigrams to stitch
PMI_MIN_FREQ = 5     # minimum corpus frequency for bigram candidates
TOP_TERMS_N  = 5     # driver/detractor terms to store per pillar per player

# Seed mode — controls which archetype set is used in Section 4
# Options:
#   "4-pillar"  — original 4 archetypes: athletic / technical / character / iq
#   "2-bin"     — binary taxonomy: god_given (ceiling) vs learned (floor)
SEED_MODE = "2-bin"

# Section weights in combined score
W_STRENGTHS  =  1.0
W_WEAKNESSES = -1.0
W_OVERVIEW   =  0

## 1. Load Data

In [99]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')

keep_cols = ['player_name', 'Pos_Group', 'position', 'grade', 'year',
             'made_it_contract', 'overview', 'strengths', 'weaknesses']
df = df[keep_cols].copy()

# Fill missing text with empty string — don't drop rows
for col in ['overview', 'strengths', 'weaknesses']:
    df[col] = df[col].fillna('')

# Drop rows with no text at all
df = df[df[['overview', 'strengths', 'weaknesses']].apply(
    lambda r: any(r.str.strip() != ''), axis=1
)].reset_index(drop=True)

print(f'Players: {len(df)}')
print(f'\nText field coverage:')
for col in ['overview', 'strengths', 'weaknesses']:
    n = (df[col].str.strip() != '').sum()
    print(f'  {col:12s}: {n:,} / {len(df):,}  ({n/len(df):.1%})')
print(f'\nPos_Group counts:')
print(df['Pos_Group'].value_counts().to_string())

Players: 6688

Text field coverage:
  overview    : 6,682 / 6,688  (99.9%)
  strengths   : 6,679 / 6,688  (99.9%)
  weaknesses  : 6,680 / 6,688  (99.9%)

Pos_Group counts:
Pos_Group
DB         1263
OL         1119
WR          933
EDGE        804
RB          616
DT          575
LB          523
TE          386
QB          300
SPECIAL     168


## 2. NFL-Aware Preprocessing

Same pipeline as `nfl_pillar_scoring.ipynb`: curated phrase stitching → domain stop words → lemmatization → PMI bigram discovery.

In [100]:
# ── Curated compound NFL terms ─────────────────────────────────────────────────
_CURATED_RAW = {
    # Trigrams (must apply before bigrams)
    'change of direction':  'change_of_direction',
    'low pad level':        'low_pad_level',
    'run after catch':      'run_after_catch',
    'yards after contact':  'yards_after_contact',
    'yards after catch':    'yards_after_catch',
    'off the line':         'off_the_line',
    'off the ball':         'off_the_ball',
    'point of attack':      'point_of_attack',
    # Bigrams
    'pass rush':            'pass_rush',
    'pass rusher':          'pass_rusher',
    'pass protection':      'pass_protection',
    'pass coverage':        'pass_coverage',
    'pad level':            'pad_level',
    'press coverage':       'press_coverage',
    'man coverage':         'man_coverage',
    'zone coverage':        'zone_coverage',
    'ball skills':          'ball_skills',
    'ball hawk':            'ball_hawk',
    'ball carrier':         'ball_carrier',
    'body control':         'body_control',
    'contact balance':      'contact_balance',
    'closing speed':        'closing_speed',
    'lateral quickness':    'lateral_quickness',
    'quick twitch':         'quick_twitch',
    'high motor':           'high_motor',
    'first step':           'first_step',
    'get off':              'get_off',
    'hand fighting':        'hand_fighting',
    'hand strength':        'hand_strength',
    'block shedding':       'block_shedding',
    'anchor strength':      'anchor_strength',
    'route running':        'route_running',
    'run blocking':         'run_blocking',
    'open field':           'open_field',
    'red zone':             'red_zone',
    'second level':         'second_level',
    'hip flexibility':      'hip_flexibility',
    'soft hands':           'soft_hands',
    'heavy hands':          'heavy_hands',
    'strong hands':         'strong_hands',
    'short area':           'short_area',
    'three down':           'three_down',
    'top end':              'top_end',
    'two gap':              'two_gap',
    'one gap':              'one_gap',
    'snap count':           'snap_count',
    # ── Added to fix stop-word collision on "game-ready" ──────────────────────
    # "game-ready" → hyphen → "game ready" → game is in CUSTOM_STOPS
    # Stitching runs BEFORE stop filter so this saves the token
    'game ready':           'game_ready',
    'hard worker':          'hard_worker',
    'work ethic':           'work_ethic',
}

CURATED_PHRASE_MAP = dict(
    sorted(_CURATED_RAW.items(), key=lambda x: len(x[0]), reverse=True)
)
print(f'Curated phrases: {len(CURATED_PHRASE_MAP)}')

Curated phrases: 49


In [101]:
# ── Domain stop words ──────────────────────────────────────────────────────────
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great', 'up', 'down',
    'off', 'out', 'over', 'through', 'above', 'below',
}

CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside', 'ceiling',
}

_base = set(stopwords.words('english'))
NFL_STOPWORDS = (_base - KEEP_WORDS) | CUSTOM_STOPS

print(f'Base NLTK stops : {len(_base)}')
print(f'Un-stopped       : {len(KEEP_WORDS & _base)}')
print(f'Custom added     : {len(CUSTOM_STOPS)}')
print(f'Final stop list  : {len(NFL_STOPWORDS)}')

Base NLTK stops : 198
Un-stopped       : 8
Custom added     : 34
Final stop list  : 224


In [102]:
lemmatizer = WordNetLemmatizer()

def nfl_preprocess(text: str, phrase_map: dict = CURATED_PHRASE_MAP,
                   extra_phrases: dict = None) -> str:
    """NFL-aware preprocessing: normalize → stitch → clean → filter → lemmatize."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)          # normalize hyphens
    for phrase, token in phrase_map.items():               # curated stitching
        text = text.replace(phrase, token)
    if extra_phrases:
        for phrase, token in extra_phrases.items():        # PMI stitching
            text = text.replace(phrase, token)
    text = re.sub(r'[^a-z_\s]', ' ', text)                # keep letters + underscores
    tokens = text.split()
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)


# Initial pass (curated phrases only — PMI added after discovery)
print('Running initial preprocessing pass...')
df['strengths_clean_v1'] = df['strengths'].apply(nfl_preprocess)
df['overview_clean_v1']  = df['overview'].apply(nfl_preprocess)
df['weaknesses_clean_v1'] = df['weaknesses'].apply(nfl_preprocess)
print('Done.')

Running initial preprocessing pass...
Done.


In [103]:
# ── PMI bigram discovery (on strengths corpus only, consistent with v1) ────────
_token_lists = [
    [t for t in text.split() if '_' not in t]
    for text in df['strengths_clean_v1']
]

finder = BigramCollocationFinder.from_documents(_token_lists)
finder.apply_freq_filter(PMI_MIN_FREQ)
scored = finder.score_ngrams(BigramAssocMeasures.pmi)

_already = set(CURATED_PHRASE_MAP.keys())
auto_candidates = [
    (w1, w2, round(score, 3), finder.ngram_fd[(w1, w2)])
    for (w1, w2), score in scored
    if  w1 not in NFL_STOPWORDS and w2 not in NFL_STOPWORDS
    and w1.isalpha() and w2.isalpha()
    and len(w1) > 2 and len(w2) > 2
    and f'{w1} {w2}' not in _already
][:PMI_TOP_N]

pmi_df = pd.DataFrame(auto_candidates, columns=['word1', 'word2', 'pmi', 'freq'])
pmi_df['phrase'] = pmi_df['word1'] + ' ' + pmi_df['word2']
pmi_df['token']  = pmi_df['word1'] + '_' + pmi_df['word2']

AUTO_PHRASE_MAP = dict(zip(pmi_df['phrase'], pmi_df['token']))
AUTO_PHRASE_MAP = dict(sorted(AUTO_PHRASE_MAP.items(), key=lambda x: len(x[0]), reverse=True))

print(f'Auto-discovered bigrams (freq ≥ {PMI_MIN_FREQ}, top {PMI_TOP_N} by PMI):')
print(pmi_df[['phrase', 'token', 'pmi', 'freq']].to_string(index=False))

Auto-discovered bigrams (freq ≥ 5, top 30 by PMI):
               phrase                 token    pmi  freq
        freight train         freight_train 15.065     8
             rag doll              rag_doll 15.065     9
          blue collar           blue_collar 14.913     7
         calling card          calling_card 14.913     7
        signal caller         signal_caller 14.913     6
       deeper deepest        deeper_deepest 14.702     7
          wake forest           wake_forest 14.691     6
           notre dame            notre_dame 14.650    12
        battering ram         battering_ram 14.623     9
       internal clock        internal_clock 14.360     6
      seeking missile       seeking_missile 13.972    10
           grows root            grows_root 13.872     7
             peek boo              peek_boo 13.843     6
          phone booth           phone_booth 13.775    20
           stat sheet            stat_sheet 13.724    15
          world class           world

In [104]:
# ── Final preprocessing — all three sections, curated + PMI phrases ────────────
print('Applying final preprocessing to all sections...')
for raw_col, clean_col in [('strengths',  'strengths_clean'),
                            ('overview',   'overview_clean'),
                            ('weaknesses', 'weaknesses_clean')]:
    df[clean_col] = df[raw_col].apply(
        lambda t: nfl_preprocess(t, extra_phrases=AUTO_PHRASE_MAP)
    )

# Drop interim v1 columns
df.drop(columns=['strengths_clean_v1', 'overview_clean_v1', 'weaknesses_clean_v1'],
        inplace=True)

all_clean = pd.concat([df['strengths_clean'], df['overview_clean'],
                       df['weaknesses_clean']])
vocab = set(t for text in all_clean for t in text.split())
stitched = [t for t in vocab if '_' in t]
print(f'Done. Vocabulary: {len(vocab):,} unique tokens ({len(stitched)} stitched phrases)')

# Spot-check
print('\nSample preprocessing (strengths):')
for _, row in df.head(2).iterrows():
    print(f"  [{row['position']}] {row['player_name']}")
    print(f"    RAW : {row['strengths'][:120]}")
    print(f"    CLEAN: {row['strengths_clean'][:120]}")
    print()

Applying final preprocessing to all sections...
Done. Vocabulary: 14,059 unique tokens (120 stitched phrases)

Sample preprocessing (strengths):
  [WR] Seyi Ajirotutu
    RAW : Ajirotutu has ideal height and a muscular body.  Long-strider with very good straight-line speed.  Fluid athlete that si
    CLEAN: ajirotutu ideal height muscular body long strider good straight line speed fluid athlete sink hip gain separation out br

  [DE] Rahim Alem
    RAW : Alem is a high-motor kid who plays to the whistle every snap.  He can be a disruptive pass rusher who sheds blockers wit
    CLEAN: alem high_motor kid play whistle every snap disruptive pass_rusher shed blocker good hand technique nose football heady 



## 3. TF-IDF Vectorization

Fit a **single vectorizer** on all three cleaned sections combined so the vocabulary and IDF weights are shared across sections. Then transform each section separately.

In [105]:
# Combine all sections for fitting
all_text_for_fit = pd.concat([
    df['strengths_clean'],
    df['overview_clean'],
    df['weaknesses_clean']
], ignore_index=True)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    token_pattern=r'[a-z_]+',   # allow underscores for stitched tokens
)
vectorizer.fit(all_text_for_fit)

mat_strengths  = vectorizer.transform(df['strengths_clean'])
mat_overview   = vectorizer.transform(df['overview_clean'])
mat_weaknesses = vectorizer.transform(df['weaknesses_clean'])

feature_names = vectorizer.get_feature_names_out()

print(f'Vocabulary size : {len(feature_names):,}')
print(f'Player matrices : {mat_strengths.shape}  (players × features)')

# Verify stitched tokens made it in
sample_tokens = ['pass_rush', 'pad_level', 'change_of_direction', 'high_motor',
                 'body_control', 'press_coverage', 'route_running', 'anchor_strength']
print('\nStitched tokens in vocabulary:')
for tok in sample_tokens:
    print(f'  {tok:30s} {"✓" if tok in vectorizer.vocabulary_ else "–"}')

Vocabulary size : 106,670
Player matrices : (6688, 106670)  (players × features)

Stitched tokens in vocabulary:
  pass_rush                      ✓
  pad_level                      ✓
  change_of_direction            ✓
  high_motor                     ✓
  body_control                   ✓
  press_coverage                 ✓
  route_running                  ✓
  anchor_strength                ✓


## 4. Pillar Archetype Definitions

Four semantic archetypes described in scout language. Each archetype is transformed using the **same fitted vectorizer** — no refitting.

In [106]:
# ── 4-Pillar archetypes (original) ────────────────────────────────────────────
ARCHETYPES_4PILLAR = {
    'score_athletic': (
        'elite athletic speed explosive burst acceleration lateral quickness '
        'body_control change_of_direction closing_speed '
        'twitch twitchy agile agility flexible flexibility fluid fluidity '
        'vertical jump leap raw physical powerful strength frame '
        'long athletic get_off first_step separation quick_twitch top_end'
    ),
    'score_technical': (
        'technique mechanics footwork hand_fighting hand_strength anchor_strength '
        'pad_level low_pad_level leverage pass_protection run_blocking '
        'route_running block_shedding press_coverage zone_coverage man_coverage '
        'precise precision timing consistent fundamental sound execution '
        'hip_flexibility balance short_area lateral_quickness polish refined '
        'crisp sharp release contested assignment scheme second_level point_of_attack'
    ),
    'score_character': (
        'high_motor motor effort compete relentless hustle competitive '
        'toughness tough grit passion energy aggressive pursuit intensity '
        'leadership leader character discipline accountable driven focused '
        'diligent committed work ethic never quit battles emotional '
        'bully mentality edge stamina workhorse warrior tireless worker '
        'blue_collar film_junkie'
    ),
    'score_iq': (
        'football iq intelligence smart instinct awareness anticipation '
        'recognition diagnosis diagnose read scheme assignment mental processing '
        'pre snap post snap understanding vision cerebral mature prepared '
        'coachable quick processor off_the_ball point_of_attack '
        'two_gap one_gap chess high football intelligence'
    ),
}

# ── 2-Bin archetypes (God-Given Ceiling vs Learned & Controlled Floor) ────────
ARCHETYPES_2BIN = {
    'score_god_given': (
        # Raw physical tools — things you can't coach
        'size height length speed twitch twitchy burst explosion '
        'natural strength frame wingspan rare specimen '
        'stamina endurance focus'
        'elite athletic powerful physical raw quick_twitch top_end '
        'vertical leap closing_speed get_off first_step '
        'body_control change_of_direction acceleration fluidity fluid '
        'agile agility flexible flexibility explosive '
        'big tall wide lean fast athlete '
    ),
    'score_learned': (
        # Core craft & technique vocabulary
        'technique hand_placement footwork leverage mechanics precision '
        'pad_level anchor_strength route_running hand_fighting '
        'press_coverage zone_coverage pass_protection run_blocking block_shedding '
        # Pass rush craft (coachable edge skills)
        'pass_rush pass_rusher shed disrupt '
        # Motor, effort, and work-ethic language
        'high_motor motor effort compete relentless hustle competitive '
        'toughness tough grit passion energy aggressive pursuit intensity '
        'work_ethic work hard hard_worker worker workhorse tireless '
        'reliable durable disciplined eager resilient effort intensity'
        # Leadership, character, discipline
        'leadership discipline accountable driven focused diligent committed '
        'blue_collar film_junkie '
        # Football intelligence & craft language
        'intelligence awareness anticipation recognition football_iq '
        'smart cerebral instinct diagnose read savvy craft '
        'scheme assignment processing coachable prepared game_ready '
        # Reliability / floor indicators
        'fundamental refined polish crisp consistent reliable dependable '
        # Standalone useful craft words scouts commonly write
        'skill skilled finish '
    ),
}

# ── Select archetype set based on SEED_MODE ────────────────────────────────────
if SEED_MODE == "2-bin":
    ARCHETYPES = ARCHETYPES_2BIN
    print(f'Seed mode: 2-BIN  (God-Given Ceiling vs Learned & Controlled Floor)')
else:
    ARCHETYPES = ARCHETYPES_4PILLAR
    print(f'Seed mode: 4-PILLAR  (Athletic / Technical / Character / IQ)')

# SCORE_COLS is derived from the selected archetypes — used by all downstream cells
SCORE_COLS = list(ARCHETYPES.keys())

# Preprocess archetype text through the same pipeline as player text
# so tokens align with the fitted vocabulary
archetype_texts  = [nfl_preprocess(ARCHETYPES[col], extra_phrases=AUTO_PHRASE_MAP)
                    for col in SCORE_COLS]
archetype_matrix = vectorizer.transform(archetype_texts)   # shape: (n_pillars, vocab)

print('Archetype vectors built:')
for col, vec, text in zip(SCORE_COLS, archetype_matrix, archetype_texts):
    print(f'  {col:30s}  non-zero features: {vec.nnz}')
print('\nSample preprocessed archetype tokens:')
for col, text in zip(SCORE_COLS, archetype_texts):
    print(f'  {col.replace("score_", ""):15s}: {text}')

Seed mode: 2-BIN  (God-Given Ceiling vs Learned & Controlled Floor)
Archetype vectors built:
  score_god_given                 non-zero features: 60
  score_learned                   non-zero features: 92

Sample preprocessed archetype tokens:
  god_given      : size height length speed twitch twitchy burst explosion natural strength frame wingspan rare specimen stamen endurance focuselite athletic powerful physical raw quick_twitch top_end vertical leap closing_speed get_off first_step body_control change_of_direction acceleration fluidity fluid agile agility flexible flexibility explosive big tall wide lean fast athlete
  learned        : technique hand_placement footwork leverage mechanic precision pad_level anchor_strength route_running hand_fighting press_coverage zone_coverage pass_protection run_blocking block_shedding pass_rush pass_rusher shed disrupt high_motor motor effort compete relentless hustle competitive toughness tough grit passion energy aggressive pursuit intensity 

## 5. Section-Level Cosine Similarity

In [107]:
sim_strengths  = cosine_similarity(mat_strengths,  archetype_matrix)   # (n, 4)
sim_weaknesses = cosine_similarity(mat_weaknesses, archetype_matrix)   # (n, 4)
sim_overview   = cosine_similarity(mat_overview,   archetype_matrix)   # (n, 4)

print('Section similarity matrices computed.')
print(f'  sim_strengths  : {sim_strengths.shape}')
print(f'  sim_weaknesses : {sim_weaknesses.shape}')
print(f'  sim_overview   : {sim_overview.shape}')

print(f'\nMean cosine sim per section:')
for name, mat in [('strengths', sim_strengths), ('weaknesses', sim_weaknesses),
                  ('overview', sim_overview)]:
    means = mat.mean(axis=0)
    print(f'  {name:12s}: ' + '  '.join(
        f'{col.replace("score_", ""):10s}={v:.4f}'
        for col, v in zip(SCORE_COLS, means)
    ))

Section similarity matrices computed.
  sim_strengths  : (6688, 2)
  sim_weaknesses : (6688, 2)
  sim_overview   : (6688, 2)

Mean cosine sim per section:
  strengths   : god_given =0.0134  learned   =0.0107
  weaknesses  : god_given =0.0093  learned   =0.0074
  overview    : god_given =0.0108  learned   =0.0103


## 6. Combined Pillar Score

```
raw = W_STRENGTHS * sim_strengths
    + W_WEAKNESSES * sim_weaknesses
    + W_OVERVIEW   * sim_overview
```
Then min-max scaled 0–100 per pillar.

In [108]:
raw_scores = (
    W_STRENGTHS  * sim_strengths +
    W_WEAKNESSES * sim_weaknesses +
    W_OVERVIEW   * sim_overview
)

scores_raw_df = pd.DataFrame(raw_scores, columns=SCORE_COLS)

scaler = MinMaxScaler(feature_range=(0, 100))
scores_scaled = pd.DataFrame(
    scaler.fit_transform(scores_raw_df),
    columns=SCORE_COLS
).round(1)

result = pd.concat(
    [df[['player_name', 'Pos_Group', 'position', 'grade', 'year',
         'made_it_contract']].reset_index(drop=True),
     scores_scaled],
    axis=1
)

print('Pillar scores computed (0–100 scaled).')
print(f'Shape: {result.shape}')
print('\nDescriptive stats:')
print(result[SCORE_COLS].describe().round(1).to_string())

Pillar scores computed (0–100 scaled).
Shape: (6688, 8)

Descriptive stats:
       score_god_given  score_learned
count           6688.0         6688.0
mean              40.8           45.1
std                9.2            9.6
min                0.0            0.0
25%               35.5           39.5
50%               40.2           44.5
75%               45.5           50.2
max              100.0          100.0


## 7. Term Attribution

For each player × pillar, the contribution of a token is:
```
contribution(player i, pillar p, token t) = tfidf(i, t) × archetype_weight(p, t)
```
This is the element-wise product of the player's TF-IDF vector and the archetype vector.

- **Drivers** = top terms from the `strengths` section (pushed score up)
- **Detractors** = top terms from the `weaknesses` section (pushed score down)

In [109]:
def top_terms_batch(contribs_dense, feature_names, top_n):
    """
    For a dense n_players × vocab array, return top_n terms per row
    by contribution score (descending), filtering to positive values only.
    Uses argpartition (O(n)) instead of full argsort (O(n log n)).
    """
    results = []
    for row in contribs_dense:
        pos_idx = np.where(row > 0)[0]
        if len(pos_idx) == 0:
            results.append([])
            continue
        n = min(top_n, len(pos_idx))
        top = pos_idx[np.argpartition(row[pos_idx], -n)[-n:]]
        top = top[np.argsort(row[top])[::-1]]
        results.append([(feature_names[j], round(float(row[j]), 4)) for j in top])
    return results


print('Building term attribution (vectorized)...')
pillar_terms_list = [{} for _ in range(len(df))]

for p_idx, pillar in enumerate(SCORE_COLS):
    arch_vec = archetype_matrix[p_idx]
    # Batch multiply: .toarray() called ONCE per section per pillar (8 total vs 26,752)
    str_contribs = mat_strengths.multiply(arch_vec).toarray()
    wk_contribs  = mat_weaknesses.multiply(arch_vec).toarray()
    drivers_all    = top_terms_batch(str_contribs, feature_names, TOP_TERMS_N)
    detractors_all = top_terms_batch(wk_contribs,  feature_names, TOP_TERMS_N)
    for i in range(len(df)):
        pillar_terms_list[i][pillar] = {
            'drivers':    drivers_all[i],
            'detractors': detractors_all[i],
        }
    print(f'  {pillar} done.')

result['pillar_terms'] = pillar_terms_list
print(f'Done. Term attribution stored for {len(result):,} players.')

Building term attribution (vectorized)...
  score_god_given done.
  score_learned done.
Done. Term attribution stored for 6,688 players.


In [110]:
# ── Flat string columns for easy reading/filtering ─────────────────────────────
# Build short names dynamically from SCORE_COLS so this works for both seed modes
PILLAR_SHORT = {col: col.replace('score_', '') for col in SCORE_COLS}

for pillar, short in PILLAR_SHORT.items():
    result[f'{short}_drivers'] = result['pillar_terms'].apply(
        lambda d, p=pillar: ', '.join(t for t, _ in d[p]['drivers'])
    )
    result[f'{short}_detractors'] = result['pillar_terms'].apply(
        lambda d, p=pillar: ', '.join(t for t, _ in d[p]['detractors'])
    )

print('Flat driver/detractor columns added.')
print('\nSample flat columns for first player:')
row = result.iloc[0]
print(f"  Player: {row['player_name']}")
for short in PILLAR_SHORT.values():
    print(f"  {short:15s} drivers   : {row[f'{short}_drivers']}")
    print(f"  {short:15s} detractors: {row[f'{short}_detractors']}")

Flat driver/detractor columns added.

Sample flat columns for first player:
  Player: Seyi Ajirotutu
  god_given       drivers   : fluid, height, body_control, athlete, speed
  god_given       detractors: explosion, agility
  learned         drivers   : 
  learned         detractors: 


## 8. Validation — Spot Check

Print a full breakdown for named players to validate the scores and term attribution make sense.

In [119]:
def player_report(name: str):
    """Print a full pillar breakdown for a named player."""
    matches = result[result['player_name'].str.contains(name, case=False, na=False)]
    if matches.empty:
        print(f'No player found matching "{name}"')
        return

    for _, row in matches.iterrows():
        idx = row.name
        print('=' * 70)
        print(f"Player : {row['player_name']}  [{row['position']} / {row['Pos_Group']}]")
        print(f"Grade  : {row.get('grade', 'N/A')}   Year: {row.get('year', 'N/A')}   "
              f"Made it: {row.get('made_it_contract', 'N/A')}")
        print()

        # Raw text sections
        print('OVERVIEW:')
        print(f"  {df.loc[idx, 'overview'][:300]}")
        print('STRENGTHS:')
        print(f"  {df.loc[idx, 'strengths'][:300]}")
        print('WEAKNESSES:')
        print(f"  {df.loc[idx, 'weaknesses'][:300]}")
        print()

        # Section-level similarity
        print(f"{'Pillar':15s} {'Strengths':>10s} {'Weaknesses':>12s} "
              f"{'Overview':>10s} {'Combined':>10s} {'Score(0-100)':>13s}")
        print('-' * 72)
        for p_idx, pillar in enumerate(SCORE_COLS):
            label = pillar.replace('score_', '')
            s_sim = sim_strengths[idx, p_idx]
            w_sim = sim_weaknesses[idx, p_idx]
            o_sim = sim_overview[idx, p_idx]
            raw   = scores_raw_df.loc[idx, pillar]
            scaled = row[pillar]
            print(f"  {label:13s} {s_sim:10.4f} {w_sim:12.4f} "
                  f"{o_sim:10.4f} {raw:10.4f} {scaled:13.1f}")
        print()

        # Term attribution
        print('TERM ATTRIBUTION:')
        terms = row['pillar_terms']
        for pillar in SCORE_COLS:
            label = pillar.replace('score_', '').upper()
            d  = terms[pillar]['drivers']
            dt = terms[pillar]['detractors']
            d_str  = ', '.join(f'{t}({s:.3f})' for t, s in d)  or '—'
            dt_str = ', '.join(f'{t}({s:.3f})' for t, s in dt) or '—'
            print(f"  {label:10s}  ▲ {d_str}")
            print(f"  {'':10s}  ▼ {dt_str}")
        print()

In [120]:
# Validate with Cam Jordan
player_report('Cameron Jordan')

Player : Cameron Jordan  [DE / EDGE]
Grade  : 7.5   Year: 2011   Made it: True

OVERVIEW:
  Jordan is one of the higher-probability, game-ready prospects in this class.  He's an ideal fit as a 3-4 defensive end but could also serve as a strongside DE in a four-man front.  Really a good fit for any team that stresses gap integrity.  Has good strength at the point of attack, plays with sound
STRENGTHS:
  Jordan is a great combination of size, strength and speed for a 3-4 defensive end prospect.  At his best against the run.  Keeps blockers off his body, has the diagnosing skills to find the football and can get off blocks and make plays.  Shows impressive stamina for a big d-end staying on the field
WEAKNESSES:
  Probably will never be an impact pass rusher, hasn’t put up big sack totals, but still works hard in that area.  Despite good bulk, may be considered a bit light in the pants for what some teams are looking for in their three-man front.  Has one documented off-the-field issue to

In [113]:
# Additional spot checks — edit names as desired
for name in ['Roquan Smith', 'Rob Gronkowski', 'Teddy Bridgewater']:
    player_report(name)

Player : Roquan Smith  [LB / LB]
Grade  : 7.0   Year: 2018   Made it: True

OVERVIEW:
  Smith is an ascending linebacker prospect with elite athletic ability, plus intelligence and an ability to be an effective cover linebacker on passing downs. While he's a little undersized, he does have the quickness and speed to keep himself from being mauled. He was good in 2016, but great in 2017
STRENGTHS:
  Former high school wideout with elite athletic ability. Speed demon who walks down backs looking to race him to the corner. Fluid and explosive in space. Block-slipper. Praised by his head coach as being a "tremendous leader" who holds himself accountable. Has instincts and football intelligence. La
WEAKNESSES:
  Is a little undersized. Has to stay one step ahead of the blocking scheme or he can be engulfed by size. Will need to diagnose and trigger just a hair faster on the next level. Occasionally comes in hot rather than breaking down in space as a tackler. Will need more schooling on han

## 9. Summary Tables

In [114]:
# Build labels dynamically from SCORE_COLS
PILLAR_LABELS = {col: col.replace('score_', '').replace('_', ' ').title() for col in SCORE_COLS}

print('TOP 10 PLAYERS PER PILLAR (with top 3 drivers)')
print('=' * 80)
for col in SCORE_COLS:
    short = PILLAR_SHORT[col]
    top = result.nlargest(10, col)[['player_name', 'position', col, f'{short}_drivers',
                                    f'{short}_detractors']].copy()
    top[f'{short}_drivers'] = top[f'{short}_drivers'].apply(
        lambda s: ', '.join(s.split(', ')[:3])  # cap at 3 for display
    )
    print(f'\n{PILLAR_LABELS[col].upper()}')
    print(top.to_string(index=False))

TOP 10 PLAYERS PER PILLAR (with top 3 drivers)

GOD GIVEN
    player_name position  score_god_given                                            god_given_drivers god_given_detractors
   Alex Parsons       OL            100.0                                  tall wide, first_step, fast             strength
    Chase Young       DE             98.9                          fluid agile, specimen, quick_twitch             athletic
  Darien Porter       CB             92.3 change_of_direction acceleration, length speed, acceleration             strength
      Jason Fox       OL             90.6                                 big tall, tall, body_control                     
 Menelik Watson       OT             88.8                               specimen, flexible, first_step           raw, frame
   Jordon Riley       DT             87.7                                        tall wide, tall, wide                     
Curtis Robinson       LB             84.8                            lengt

In [115]:
# ── Per-player % composition (add to result) ───────────────────────────────────
# Clip raw scores at 0 before row-normalizing (weighted formula can go negative)
raw_clipped = scores_raw_df.clip(lower=0)
row_totals  = raw_clipped.sum(axis=1).replace(0, np.nan)  # avoid div-by-zero
pct_df = raw_clipped.div(row_totals, axis=0).mul(100).round(1)
pct_df.columns = [f'{col}_pct' for col in SCORE_COLS]
result = pd.concat([result, pct_df], axis=1)

# ── Median scores by position group ────────────────────────────────────────────
pct_cols = list(pct_df.columns)
median_scores = (
    result.groupby('Pos_Group')[SCORE_COLS + pct_cols]
    .median()
    .round(1)
    .sort_values(SCORE_COLS[0], ascending=False)
)

short_rename = {col: col.replace('score_', '') for col in SCORE_COLS}
short_rename.update({f'{col}_pct': f'{col.replace("score_", "")}_pct%' for col in SCORE_COLS})

print('Median Pillar Scores by Position Group')
print('Left columns: 0-100 (MinMax across all players) | Right columns: % mix per player')
print(median_scores.rename(columns=short_rename).to_string())

Median Pillar Scores by Position Group
Left columns: 0-100 (MinMax across all players) | Right columns: % mix per player
           god_given  learned  god_given_pct%  learned_pct%
Pos_Group                                                  
TE              41.6     44.1            69.6          30.4
WR              41.2     44.2            71.6          28.4
EDGE            41.0     44.8            63.6          36.4
DB              40.7     45.0            54.7          45.3
DT              40.7     44.5            56.0          44.0
RB              40.2     44.3            58.0          42.0
QB              39.7     42.8            52.5          47.5
OL              39.3     44.5            46.6          53.4
LB              38.7     46.1            23.0          77.0
SPECIAL         38.3     42.7            47.8          52.2


In [116]:
# ── Pooled positional profile — % composition (strengths vs weaknesses) ────────
# Pool strengths and weaknesses text separately by position group
pooled_str = (
    df.groupby('Pos_Group')['strengths_clean']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
)
pooled_wk = (
    df.groupby('Pos_Group')['weaknesses_clean']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
)

str_mat = vectorizer.transform(pooled_str['strengths_clean'])
wk_mat  = vectorizer.transform(pooled_wk['weaknesses_clean'])

pos_index = pooled_str['Pos_Group']

# Raw cosine sims (always non-negative — safe to row-normalize)
str_sims = pd.DataFrame(cosine_similarity(str_mat, archetype_matrix),
                        columns=SCORE_COLS, index=pos_index)
wk_sims  = pd.DataFrame(cosine_similarity(wk_mat,  archetype_matrix),
                        columns=SCORE_COLS, index=pos_index)

# Row-normalize to % composition: each position row sums to 100
strengths_pct = str_sims.div(str_sims.sum(axis=1), axis=0).mul(100).round(1)
weakness_pct  = wk_sims.div(wk_sims.sum(axis=1),  axis=0).mul(100).round(1)

# Gap: strengths% - weakness% per bin (positive = praised more than critiqued)
gap_pct = (strengths_pct - weakness_pct).round(1)

# Rename columns for display
short_map = {col: col.replace('score_', '') for col in SCORE_COLS}
sort_col  = SCORE_COLS[0]  # sort by first bin

print('POSITIONAL ARCHETYPE MIX — STRENGTHS (% of praise by archetype)')
print('Each row sums to 100%')
print(strengths_pct.rename(columns=short_map)
      .sort_values(short_map[sort_col], ascending=False).to_string())

print('\nPOSITIONAL ARCHETYPE MIX — WEAKNESSES (% of critique by archetype)')
print('Each row sums to 100%')
print(weakness_pct.rename(columns=short_map)
      .sort_values(short_map[sort_col], ascending=False).to_string())

print('\nGAP (Strengths% − Weakness%) — positive = praised more than critiqued for this archetype')
print(gap_pct.rename(columns=short_map)
      .sort_values(short_map[sort_col], ascending=False).to_string())

POSITIONAL ARCHETYPE MIX — STRENGTHS (% of praise by archetype)
Each row sums to 100%
           god_given  learned
Pos_Group                    
DT              49.9     50.1
WR              48.9     51.1
TE              48.8     51.2
DB              48.2     51.8
EDGE            47.8     52.2
OL              45.0     55.0
RB              44.7     55.3
LB              42.7     57.3
SPECIAL         38.8     61.2
QB              36.6     63.4

POSITIONAL ARCHETYPE MIX — WEAKNESSES (% of critique by archetype)
Each row sums to 100%
           god_given  learned
Pos_Group                    
SPECIAL         51.7     48.3
DT              49.9     50.1
DB              49.7     50.3
WR              49.4     50.6
LB              49.0     51.0
OL              48.9     51.1
EDGE            46.4     53.6
RB              45.9     54.1
TE              43.0     57.0
QB              35.6     64.4

GAP (Strengths% − Weakness%) — positive = praised more than critiqued for this archetype
           god

In [117]:
# ── Save output ────────────────────────────────────────────────────────────────
short_names = [col.replace('score_', '') for col in SCORE_COLS]
pct_cols    = [f'{col}_pct' for col in SCORE_COLS]

flat_cols = (
    ['player_name', 'Pos_Group', 'position', 'grade', 'year', 'made_it_contract']
    + SCORE_COLS
    + pct_cols
    + [f'{s}_{d}' for s in short_names for d in ['drivers', 'detractors']]
)

out_path = f'../data/processed/pillar_scores_v2_{SEED_MODE.replace("-", "_")}.csv'
result[flat_cols].to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(result):,} rows, {len(flat_cols)} columns)')
print('\nColumn list:')
for c in flat_cols:
    print(f'  {c}')

Saved: ../data/processed/pillar_scores_v2_2_bin.csv  (6,688 rows, 14 columns)

Column list:
  player_name
  Pos_Group
  position
  grade
  year
  made_it_contract
  score_god_given
  score_learned
  score_god_given_pct
  score_learned_pct
  god_given_drivers
  god_given_detractors
  learned_drivers
  learned_detractors
